In [1]:
!pip install -U transformers datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 107.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 113.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 89.0 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: datasets
    Found existing installati

In [6]:
# ------------------------------------------------------------
# 0) Install
# ------------------------------------------------------------
# !pip install -U transformers datasets scikit-learn

# ------------------------------------------------------------
# 1) Imports
# ------------------------------------------------------------
from transformers import AutoTokenizer, AutoModel
from transformers import TrainingArguments, Trainer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

# ------------------------------------------------------------
# 2) Load Data
# ------------------------------------------------------------
df = pd.read_csv("/kaggle/input/datasets/shuvo128/sasyfy/IMDB_Cleaned (1).csv")

text_column = "cleaned_review" if "cleaned_review" in df.columns else "review"
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

texts  = df[text_column].astype(str).tolist()
labels = (df["sentiment"] == "positive").astype(int).tolist()

print("Labels:", set(labels))

# ------------------------------------------------------------
# 3) Split
# ------------------------------------------------------------
train_texts = texts[:40000]
train_labels = labels[:40000]

val_texts = texts[40000:45000]
val_labels = labels[40000:45000]

test_texts = texts[45000:50000]
test_labels = labels[45000:50000]

# ------------------------------------------------------------
# 4) TF-IDF (FIXED)
# ------------------------------------------------------------
tfidf = TfidfVectorizer(max_features=5000, norm='l2')
tfidf.fit(train_texts)

train_tfidf = tfidf.transform(train_texts).toarray()
val_tfidf   = tfidf.transform(val_texts).toarray()
test_tfidf  = tfidf.transform(test_texts).toarray()

# ------------------------------------------------------------
# 5) Tokenizer
# ------------------------------------------------------------
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts):
    return tokenizer(texts, truncation=True, padding=True, max_length=256)

train_enc = tokenize(train_texts)
val_enc   = tokenize(val_texts)
test_enc  = tokenize(test_texts)

# ------------------------------------------------------------
# 6) Dataset
# ------------------------------------------------------------
class HybridDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, tfidf_features, labels):
        self.encodings = encodings
        self.tfidf = tfidf_features
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["tfidf"] = torch.tensor(self.tfidf[idx], dtype=torch.float32)
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HybridDataset(train_enc, train_tfidf, train_labels)
val_dataset   = HybridDataset(val_enc, val_tfidf, val_labels)
test_dataset  = HybridDataset(test_enc, test_tfidf, test_labels)

# ------------------------------------------------------------
# 7) MODEL (FULL FIX)
# ------------------------------------------------------------
class DebertaTFIDF(nn.Module):
    def __init__(self, model_name, tfidf_dim, num_labels):
        super().__init__()
        self.deberta = AutoModel.from_pretrained(model_name)
        hidden_size = self.deberta.config.hidden_size

        # TF-IDF projection (balanced)
        self.tfidf_proj = nn.Sequential(
            nn.Linear(tfidf_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Fusion normalization
        self.fusion_norm = nn.LayerNorm(hidden_size + 256)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + 256, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_labels)
        )

    def mean_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = torch.sum(last_hidden_state * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / counts

    def forward(self, input_ids, attention_mask, tfidf, labels=None):
        outputs = self.deberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = self.mean_pooling(outputs.last_hidden_state, attention_mask)

        tfidf_out = self.tfidf_proj(tfidf)

        # 🔥 critical scaling
        tfidf_out = 0.3 * tfidf_out

        combined = torch.cat((pooled, tfidf_out), dim=1)
        combined = self.fusion_norm(combined)

        logits = self.classifier(combined)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

model = DebertaTFIDF(
    model_name=MODEL_NAME,
    tfidf_dim=5000,
    num_labels=2
)

# ------------------------------------------------------------
# 8) Metrics
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# ------------------------------------------------------------
# 9) Training Args (STABLE)
# ------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-6,
    weight_decay=0.01,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    logging_steps=100,
    save_strategy="no",
    report_to="none",
    max_grad_norm=1.0
)

# ------------------------------------------------------------
# 10) Trainer
# ------------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# ------------------------------------------------------------
# 11) Train
# ------------------------------------------------------------
trainer.train()

# ------------------------------------------------------------
# 12) Evaluate
# ------------------------------------------------------------
print("\nValidation:")
print(trainer.evaluate())

print("\nTest:")
test_results = trainer.predict(test_dataset)

test_preds = np.argmax(test_results.predictions, axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average='binary'
)
acc = accuracy_score(test_labels, test_preds)

print({
    "accuracy": acc,
    "precision": precision,
    "recall": recall,
    "f1": f1
})

Labels: {0, 1}


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-package

Step,Training Loss
100,0.711887
200,0.000000
300,0.000000
400,0.000000
500,0.000000
600,0.000000
700,0.000000
800,0.000000
900,0.000000
1000,0.000000


KeyboardInterrupt: 